# Football Analysis Pipeline

---

Analyze broadcast football footage using computer vision and machine learning. This pipeline detects players, goalkeepers, referees, and the ball, tracks their movements, classifies teams by jersey color, and generates annotated video output.

![Football AI](https://media.roboflow.com/notebooks/examples/football-ai-diagram.png)

### What This Notebook Does

| Stage | Technology | Output |
|-------|------------|--------|
| **Detection** | YOLOv8 | Players, GK, referees, ball |
| **Tracking** | ByteTrack | Unique IDs across frames |
| **Team Classification** | SigLIP + KMeans | Team colors (red vs blue) |
| **Ball Tracking** | Slicer + Kalman | Smooth ball trajectory |
| **Output** | Supervision | Annotated video file |

## Before You Start

### Select GPU Runtime

**Note:** Processing is 10-50x faster with GPU. Navigate to `Runtime` > `Change runtime type` > `Hardware accelerator` > `GPU (T4)` and click `Save`.

In [ ]:
!nvidia-smi

## Install Dependencies

**Note:** This cell clones the repository, installs packages, and downloads pre-trained models. Takes 2-3 minutes on first run.

In [ ]:
import os
import shutil
from pathlib import Path

REPO_URL = "https://github.com/esharif20/Spatio-Temporal-GNN-Football-Analysis.git"
REPO_DIR = "/content/football_analysis"

os.chdir("/content")
if Path(REPO_DIR).exists():
    shutil.rmtree(REPO_DIR)

print("Cloning repository...")
!git clone --quiet {REPO_URL} {REPO_DIR}
os.chdir(REPO_DIR)

print("Installing dependencies...")
os.environ["TF_CPP_MIN_LOG_LEVEL"] = "3"
!pip install -q "numpy<2"
!bash colab_setup.sh 2>&1 | grep -E "(Downloading|Available|Example)" | head -15

print("\n" + "="*50)
print("SETUP COMPLETE")
print("="*50)

### Verify Environment

In [ ]:
import torch
from pathlib import Path

print("="*50)
print("ENVIRONMENT")
print("="*50)

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Device: {DEVICE}")
if DEVICE == "cuda":
    print(f"GPU: {torch.cuda.get_device_name(0)}")

print("\nModels:")
for name, path in [("Player", "src/models/player_detection.pt"),
                   ("Ball", "src/models/ball_detection.pt"),
                   ("Pitch", "src/models/pitch_detection.pt")]:
    status = "OK" if Path(path).exists() else "MISSING"
    print(f"  {name}: {status}")

print("\nSample Videos:")
videos = sorted(Path("src/input_videos").glob("*.mp4"))
for v in videos[:5]:
    size = v.stat().st_size / (1024*1024)
    print(f"  {v.stem} ({size:.1f} MB)")

---

## Pipeline Configuration

| Mode | What It Does | Speed |
|------|--------------|-------|
| `all` | Detection + Tracking + Teams + Ball + Pitch Keypoints | Slowest |
| `team` | Detection + Tracking + Team colors | Medium |
| `track` | Detection + Tracking (no team colors) | Medium |
| `players` | Detection only (bounding boxes) | Fast |
| `ball` | Ball detection and tracking only | Medium |
| `pitch` | Pitch keypoint detection only | Fast |
| `radar` | 2D tactical top-down view with homography | Slowest |

In [ ]:
#@title Configuration { display-mode: "form" }

#@markdown ### Video & Mode
CLIP = "0bfacc_0"  #@param ["0bfacc_0", "2e57b9_0", "08fd33_0", "573e61_0", "121364_0"]
MODE = "all"  #@param ["all", "team", "track", "players", "ball", "pitch", "radar"]

#@markdown ### Options
BALL_CONF = 0.15  #@param {type:"slider", min:0.05, max:0.5, step:0.05}
FAST_BALL = False  #@param {type:"boolean"}
FRESH = True  #@param {type:"boolean"}

print(f"Clip: {CLIP} | Mode: {MODE} | Ball conf: {BALL_CONF}")

---

## Run Pipeline

**Note:** Processing time: 2-5 minutes for a 30-second clip.

In [ ]:
import os
import torch
from pathlib import Path

# Validate
clip_path = Path(f"src/input_videos/{CLIP}.mp4")
if not clip_path.exists():
    available = [p.stem for p in Path("src/input_videos").glob("*.mp4")]
    raise FileNotFoundError(f"Video '{CLIP}' not found.\nAvailable: {available}")

device = "cuda" if torch.cuda.is_available() else "cpu"

# Build command
cmd = f"DEVICE={device} bash src/run.sh {MODE} {CLIP}"
if FRESH:
    cmd += " --fresh"
if FAST_BALL:
    cmd += " --fast-ball"
else:
    cmd += f" --ball-conf {BALL_CONF}"

print("="*60)
print(f"Running: {MODE} mode on {CLIP}")
print(f"Device: {device}")
print("="*60 + "\n")

os.environ["TF_CPP_MIN_LOG_LEVEL"] = "3"
!{cmd}

# Set paths
OUTPUT_VIDEO = f"src/output_videos/{CLIP}/{CLIP}_{MODE.upper()}.mp4"
INPUT_VIDEO = f"src/input_videos/{CLIP}.mp4"

if Path(OUTPUT_VIDEO).exists():
    size = Path(OUTPUT_VIDEO).stat().st_size / (1024*1024)
    print(f"\n" + "="*60)
    print(f"OUTPUT: {OUTPUT_VIDEO} ({size:.1f} MB)")
    print("="*60)

---

## Visualize Results

### Preview Output Frames

**Note:** Shows key frames from the processed video.

In [ ]:
import cv2
import matplotlib.pyplot as plt
from pathlib import Path

def extract_frames(video_path, num_frames=5):
    """Extract evenly spaced frames from video."""
    cap = cv2.VideoCapture(str(video_path))
    total = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    indices = [int(i * total / num_frames) for i in range(num_frames)]
    
    frames = []
    for idx in indices:
        cap.set(cv2.CAP_PROP_POS_FRAMES, idx)
        ret, frame = cap.read()
        if ret:
            frames.append(cv2.cvtColor(frame, cv2.COLOR_BGR2RGB))
    cap.release()
    return frames, indices

if Path(OUTPUT_VIDEO).exists():
    frames, indices = extract_frames(OUTPUT_VIDEO, num_frames=4)
    
    fig, axes = plt.subplots(2, 2, figsize=(14, 8))
    for ax, frame, idx in zip(axes.flat, frames, indices):
        ax.imshow(frame)
        ax.set_title(f"Frame {idx}", fontsize=10)
        ax.axis("off")
    plt.suptitle(f"Output: {CLIP}_{MODE.upper()}.mp4", fontsize=12, fontweight="bold")
    plt.tight_layout()
    plt.show()
else:
    print("Output video not found. Run the pipeline first.")

### Before & After Comparison

In [ ]:
import cv2
import matplotlib.pyplot as plt
from pathlib import Path

def get_frame(video_path, frame_idx):
    cap = cv2.VideoCapture(str(video_path))
    cap.set(cv2.CAP_PROP_POS_FRAMES, frame_idx)
    ret, frame = cap.read()
    cap.release()
    return cv2.cvtColor(frame, cv2.COLOR_BGR2RGB) if ret else None

if Path(INPUT_VIDEO).exists() and Path(OUTPUT_VIDEO).exists():
    # Get frame from middle of video
    cap = cv2.VideoCapture(OUTPUT_VIDEO)
    mid_frame = int(cap.get(cv2.CAP_PROP_FRAME_COUNT) / 2)
    cap.release()
    
    input_frame = get_frame(INPUT_VIDEO, mid_frame)
    output_frame = get_frame(OUTPUT_VIDEO, mid_frame)
    
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 6))
    ax1.imshow(input_frame)
    ax1.set_title("Original", fontsize=12, fontweight="bold")
    ax1.axis("off")
    ax2.imshow(output_frame)
    ax2.set_title("Processed", fontsize=12, fontweight="bold")
    ax2.axis("off")
    plt.suptitle(f"Frame {mid_frame}", fontsize=10)
    plt.tight_layout()
    plt.show()
else:
    print("Videos not found.")

### Detection Statistics

In [ ]:
import pickle
from pathlib import Path

stub_dir = Path("src/stubs")

print("="*50)
print("DETECTION STATISTICS")
print("="*50)

# People tracks
people_stub = stub_dir / f"{CLIP}_people_tracks.pkl"
if people_stub.exists():
    with open(people_stub, "rb") as f:
        people = pickle.load(f)
    
    for category in ["players", "goalkeepers", "referees"]:
        if category in people:
            ids = set()
            for frame_data in people[category]:
                ids.update(frame_data.keys())
            print(f"{category.capitalize()}: {len(ids)} unique")
else:
    print("No tracking data found.")

# Ball tracks
ball_stub = stub_dir / f"{CLIP}_ball_tracks.pkl"
if ball_stub.exists():
    with open(ball_stub, "rb") as f:
        ball = pickle.load(f)
    if "ball" in ball:
        detected = sum(1 for f in ball["ball"] if f)
        total = len(ball["ball"])
        print(f"Ball: {detected}/{total} frames ({100*detected/total:.0f}%)")

print("="*50)

---

## Download Output

**Note:** Click the link below to download the video file.

In [ ]:
from pathlib import Path
import shutil

if Path(OUTPUT_VIDEO).exists():
    # Copy to /content for easy download
    output_name = f"{CLIP}_{MODE.upper()}.mp4"
    download_path = f"/content/{output_name}"
    shutil.copy(OUTPUT_VIDEO, download_path)
    
    size = Path(download_path).stat().st_size / (1024*1024)
    print(f"Video ready: {output_name} ({size:.1f} MB)")
    print(f"\nDownload from: Files panel (left sidebar) > {output_name}")
    print("Or right-click the file and select 'Download'")
else:
    print("No output video found.")

---

## Upload Your Own Video

In [ ]:
from google.colab import files
from pathlib import Path
import shutil

print("Select a video file (MP4):")
uploaded = files.upload()

for filename in uploaded.keys():
    dest = Path(f"src/input_videos/{filename}")
    shutil.move(filename, dest)
    print(f"\nUploaded: {dest.stem}")
    print(f"Set CLIP = \"{dest.stem}\" above and re-run.")

---

## Reference

### CLI Options

| Option | Description |
|--------|-------------|
| `--fresh` | Reprocess everything (ignore cache) |
| `--fast-ball` | Faster ball tracking (less accurate) |
| `--ball-conf 0.10` | Lower = more ball detections |
| `--no-ball-model` | Use multi-class model for ball |

### Troubleshooting

| Problem | Solution |
|---------|----------|
| No GPU | `Runtime` > `Change runtime type` > `GPU` |
| numpy error | `Runtime` > `Restart runtime` |
| Ball not detected | Lower `BALL_CONF` to 0.10 |

### Links

- [GitHub](https://github.com/esharif20/Spatio-Temporal-GNN-Football-Analysis)
- [YOLOv8](https://docs.ultralytics.com/)
- [Supervision](https://supervision.roboflow.com/)